# Appendix-Style Figure Workflows

This notebook is one chapter of the runnable `KnottedGraph` user guide.  It is
generated into `User_guide/10_appendix_workflows.ipynb` so users can open the specific workflow
they need without navigating one very large notebook.

- self-contained after the shared setup cells


## 0. Setup, Preflight, And Shared Plot Style

The whole notebook uses the same visual convention:

- blue: surfaces, skeleton points, and graph edges;
- red: graph vertices;
- black axes;
- paper notation: `Upsilon(G; Y)`.

The helper functions in this section remove repeated plotting boilerplate from
the rest of the notebook.  This is also the library-level pattern worth
promoting later into public visualization helpers.


In [1]:
from pathlib import Path
import sys
import importlib.util
import os
import tempfile

PROJECT_ROOT = Path.cwd()
while not (PROJECT_ROOT / "src").exists() and PROJECT_ROOT != PROJECT_ROOT.parent:
    PROJECT_ROOT = PROJECT_ROOT.parent

DOC_ROOT = PROJECT_ROOT / "doc"
SRC_ROOT = PROJECT_ROOT / "src"
if str(SRC_ROOT) not in sys.path:
    sys.path.insert(0, str(SRC_ROOT))

os.environ.setdefault("MPLCONFIGDIR", str(Path(tempfile.gettempdir()) / "knottedgraph-mpl"))

print("project paths configured")
for package in ["numpy", "networkx", "sympy", "plotly", "matplotlib", "pyvista"]:
    print(f"{package:10s} = {importlib.util.find_spec(package) is not None}")


project paths configured
numpy      = True
networkx   = True
sympy      = True
plotly     = True
matplotlib = True
pyvista    = True


In [2]:
import math
import time

import networkx as nx
import numpy as np
import matplotlib.pyplot as plt
import plotly.graph_objects as go
import plotly.io as pio
import sympy as sp
from IPython.display import Math, display

from knotted_graph.projection import (
    compute_yamada_polynomial,
    sample_projections,
    select_projection,
)
from knotted_graph.visualization import plot_3D_graph_plotly

BLUE = "#1f77b4"
RED = "#d62728"
CAMERA = dict(eye=dict(x=1.45, y=1.55, z=1.18))
pio.renderers.default = "notebook_connected"
Y = sp.Symbol("Y")
kx, ky, kz = sp.symbols("k_x k_y k_z", real=True)


def axis_style():
    return dict(
        visible=True,
        title="",
        showticklabels=False,
        showbackground=False,
        showgrid=False,
        zeroline=False,
        showline=True,
        linecolor="black",
        linewidth=2,
    )


def apply_kg_layout(fig, *, width=760, height=620):
    fig.update_layout(
        title=None,
        width=width,
        height=height,
        margin=dict(l=0, r=0, t=0, b=0),
        scene=dict(
            xaxis=axis_style(),
            yaxis=axis_style(),
            zaxis=axis_style(),
            aspectmode="data",
            camera=CAMERA,
        ),
    )
    return fig


def plot_surface_polydata(surface, *, opacity=0.58):
    mesh = surface.triangulate()
    faces = mesh.faces.reshape(-1, 4)[:, 1:]
    pts = mesh.points
    fig = go.Figure(
        go.Mesh3d(
            x=pts[:, 0],
            y=pts[:, 1],
            z=pts[:, 2],
            i=faces[:, 0],
            j=faces[:, 1],
            k=faces[:, 2],
            color=BLUE,
            opacity=opacity,
        )
    )
    return apply_kg_layout(fig)


def plot_points_3d(points, *, size=3):
    points = np.asarray(points)
    fig = go.Figure(
        go.Scatter3d(
            x=points[:, 0],
            y=points[:, 1],
            z=points[:, 2],
            mode="markers",
            marker=dict(size=size, color=BLUE),
        )
    )
    return apply_kg_layout(fig)


def plot_graph_kg(graph):
    return apply_kg_layout(plot_3D_graph_plotly(graph))


def print_upsilon(label, expr):
    print(f"Upsilon({label}; Y) = {sp.expand(expr)}")


def display_bloch_vector(label, components):
    display(Math(label + r"=" + sp.latex(sp.Matrix(components))))


print("shared plotting and notation helpers ready")


shared plotting and notation helpers ready


In [3]:
from knotted_graph.applications.nodal import NodalSkeleton
from knotted_graph.applications.nodal.models import (
    awesome_bloch_vector,
    hopf_link_bloch_vector,
    pq_torus_knot_bloch_vector,
    solomon_bloch_vector,
    threelink_bloch_vector,
    trefoil_bloch_vector,
    unknot_bloch_vector,
)

print("nodal application imports ready")


nodal application imports ready


## 10. Appendix-Style Figure Workflows

The cells below build manuscript-style scientific panels directly from
`KnottedGraph` objects: surfaces, skeleton points, spatial graphs, projections,
Yamada fingerprints, Berry slices, minor checks, and planarity changes.


In [4]:
from plotly.subplots import make_subplots
from knotted_graph.applications.nodal.models import (
    awesome_bloch_vector,
    hopf_link_bloch_vector,
    pq_torus_knot_bloch_vector,
    solomon_bloch_vector,
    threelink_bloch_vector,
    trefoil_bloch_vector,
    unknot_bloch_vector,
)

def add_surface_trace(fig, surface, *, row=1, col=1, opacity=0.58, color=BLUE):
    mesh = surface.triangulate()
    faces = mesh.faces.reshape(-1, 4)[:, 1:]
    pts = mesh.points
    fig.add_trace(
        go.Mesh3d(
            x=pts[:, 0],
            y=pts[:, 1],
            z=pts[:, 2],
            i=faces[:, 0],
            j=faces[:, 1],
            k=faces[:, 2],
            color=color,
            opacity=opacity,
            showscale=False,
        ),
        row=row,
        col=col,
    )


def add_points_trace(fig, points, *, row=1, col=1, size=2.5, color=BLUE):
    points = np.asarray(points)
    fig.add_trace(
        go.Scatter3d(
            x=points[:, 0],
            y=points[:, 1],
            z=points[:, 2],
            mode="markers",
            marker=dict(size=size, color=color),
            showlegend=False,
        ),
        row=row,
        col=col,
    )


def add_graph_traces(fig, graph, *, row=1, col=1):
    graph_fig = plot_3D_graph_plotly(graph)
    for trace in graph_fig.data:
        fig.add_trace(trace, row=row, col=col)


def style_plotly_scenes(fig, scene_count, *, width=980, height=660):
    for index in range(1, scene_count + 1):
        scene_name = "scene" if index == 1 else f"scene{index}"
        fig.update_layout(
            **{
                scene_name: dict(
                    xaxis=axis_style(),
                    yaxis=axis_style(),
                    zaxis=axis_style(),
                    aspectmode="data",
                    camera=CAMERA,
                )
            }
        )
    fig.update_layout(
        title=None,
        width=width,
        height=height,
        margin=dict(l=0, r=0, t=0, b=0),
        showlegend=False,
    )
    return fig


def polyline_traces_from_slice(polydata):
    points = np.asarray(polydata.points)
    lines = np.asarray(polydata.lines)
    traces = []
    cursor = 0
    while cursor < len(lines):
        n = int(lines[cursor])
        ids = lines[cursor + 1 : cursor + 1 + n]
        cursor += n + 1
        if n < 2:
            continue
        pts = points[ids]
        traces.append(pts)
    return traces

print("appendix plotting helpers ready")


appendix plotting helpers ready


### 10.1 Skeletonization Appendix: Torus `(2,4)`

The original appendix showed two thicknesses for a `(2,4)` torus model.  In the
current public extraction path, `gamma=0.06` still produces a surface and
skeleton points but the simplified graph collapses; `gamma=0.2` produces the
full graph stage.  Showing both is more useful than hiding the fragile case.


In [5]:
appendix_torus_records = []
for gamma in (0.06, 0.2):
    ske_t24 = NodalSkeleton(
        pq_torus_knot_bloch_vector(2, 4, gamma, k_symbols=(kx, ky, kz), c=0.7, m=2.0),
        k_symbols=(kx, ky, kz),
        dimension=120,
        axis_scale=(1.0, 1.0, 1.5),
    )
    surface_t24 = ske_t24.exceptional_surface_pv.connectivity("largest")
    points_t24 = ske_t24.skeleton_coords
    graph_t24 = None
    graph_error = None
    try:
        graph_t24 = ske_t24.skeleton_graph(simplify=True, smooth_epsilon=2)
    except Exception as exc:
        graph_error = f"{type(exc).__name__}: {exc}"
    appendix_torus_records.append((gamma, surface_t24, points_t24, graph_t24, graph_error))
    print(f"gamma = {gamma}")
    print("  surface_points_cells =", (surface_t24.n_points, surface_t24.n_cells))
    print("  skeleton_points =", points_t24.shape)
    if graph_t24 is None:
        print("  graph_stage =", graph_error)
    else:
        print("  graph_nodes_edges =", (graph_t24.number_of_nodes(), graph_t24.number_of_edges()))


gamma = 0.06
  surface_points_cells = (876, 1748)
  skeleton_points = (381, 3)
  graph_stage = EmbeddingValidationError: graph has no edges


gamma = 0.2
  surface_points_cells = (13704, 27420)
  skeleton_points = (579, 3)
  graph_nodes_edges = (2, 5)


In [6]:
fig = make_subplots(
    rows=1,
    cols=2,
    specs=[[{"type": "scene"}, {"type": "scene"}]],
    horizontal_spacing=0.02,
)
for col, (gamma, surface_t24, points_t24, graph_t24, graph_error) in enumerate(appendix_torus_records, start=1):
    add_surface_trace(fig, surface_t24, row=1, col=col, opacity=0.42)
    add_points_trace(fig, points_t24, row=1, col=col, size=2.3, color=RED if graph_t24 is None else BLUE)
style_plotly_scenes(fig, 2, width=980, height=520).show()


In [7]:
gamma, surface_t24, points_t24, graph_t24, graph_error = appendix_torus_records[1]
fig = plot_graph_kg(graph_t24)
fig.show()


### 10.2 Yamada Appendix Table

The appendix table compares knot/graph families across parameter choices.  The
cell below keeps all rows from the appendix schedule.  When a row does not
produce a valid graph/projection at this practical notebook resolution, the
failure is printed explicitly; that tells the user to inspect thickness,
resolution, simplification, and projection choice before trusting a polynomial.


In [8]:
appendix_yamada_specs = [
    ("Hopf link", "Y^2 + 1", hopf_link_bloch_vector, [0.1, 0.2, 0.5], 64),
    ("Trefoil", "Y - 1 + 1/Y", trefoil_bloch_vector, [0.1, 0.19, 0.25], 64),
    ("Torus (1,2)", "1", lambda gamma, k_symbols: pq_torus_knot_bloch_vector(1, 2, gamma, k_symbols=k_symbols), [0.12, 0.5, 0.7], 64),
    ("Solomon", "Y^2 - Y + 1 - 1/Y - 1/Y^2", solomon_bloch_vector, [0.12, 1.0, 2.0], 64),
]

appendix_yamada_rows = []
for family, classical_reference, builder, gammas, dimension in appendix_yamada_specs:
    for gamma in gammas:
        row = {
            "family": family,
            "gamma": gamma,
            "classical_reference": classical_reference,
            "dimension": dimension,
            "status": "ok",
        }
        try:
            ske_row = NodalSkeleton(
                builder(gamma, k_symbols=(kx, ky, kz)),
                k_symbols=(kx, ky, kz),
                dimension=dimension,
                axis_scale=(1.0, 1.0, 1.5),
            )
            graph_row = ske_row.skeleton_graph(simplify=True, smooth_epsilon=2)
            projection_row = select_projection(graph_row, num_rotation_samples=8)
            polynomial_row = compute_yamada_polynomial(
                graph_row,
                Y,
                rotation_angles=projection_row.rotation_angles,
                n_jobs=1,
            )
            row.update(
                nodes=graph_row.number_of_nodes(),
                edges=graph_row.number_of_edges(),
                crossings=projection_row.num_crossings,
                upsilon=str(sp.expand(polynomial_row)),
            )
        except Exception as exc:
            row.update(status="inspect", reason=f"{type(exc).__name__}: {str(exc)[:120]}")
        appendix_yamada_rows.append(row)

for row in appendix_yamada_rows:
    print(f"{row['family']:12s} gamma={row['gamma']:<4} status={row['status']}")
    print(f"  classical_reference = {row['classical_reference']}")
    if row["status"] == "ok":
        print(f"  nodes_edges_crossings = {(row['nodes'], row['edges'], row['crossings'])}")
        print(f"  Upsilon(G; Y) = {row['upsilon']}")
    else:
        print(f"  needs inspection = {row['reason']}")


Hopf link    gamma=0.1  status=inspect
  classical_reference = Y^2 + 1
  needs inspection = EmbeddingValidationError: graph has no edges
Hopf link    gamma=0.2  status=ok
  classical_reference = Y^2 + 1
  nodes_edges_crossings = (4, 6, 2)
  Upsilon(G; Y) = Y**10 + Y**9 + Y**8 + 2*Y**7 + 2*Y**5 + 2*Y**3 + Y**2 + Y + 1
Hopf link    gamma=0.5  status=ok
  classical_reference = Y^2 + 1
  nodes_edges_crossings = (2, 3, 1)
  Upsilon(G; Y) = -Y**4 - Y**3 - 2*Y**2 - Y - 1
Trefoil      gamma=0.1  status=inspect
  classical_reference = Y - 1 + 1/Y
  needs inspection = RuntimeError: All projection samples failed: sample 0: Found overlapping (colinear) segments; sample 1: Found overlapping (colinear) s
Trefoil      gamma=0.19 status=ok
  classical_reference = Y - 1 + 1/Y
  nodes_edges_crossings = (1, 1, 0)
  Upsilon(G; Y) = -Y**2 - Y - 1
Trefoil      gamma=0.25 status=ok
  classical_reference = Y - 1 + 1/Y
  nodes_edges_crossings = (3, 5, 3)
  Upsilon(G; Y) = -Y**6 - Y**5 - 3*Y**4 - 2*Y**3 - 3*Y**

### 10.3 Energy-Isosurface Appendix Gallery

The appendix gallery compared several Hamiltonian families.  This notebook
rebuilds the surfaces and extracted spatial graphs from the model functions,
then prints which panels are planar or non-planar after simplification.


In [9]:
energy_gallery_specs = [
    ("Unknot", lambda: unknot_bloch_vector(0.1, k_symbols=(kx, ky, kz)), 64),
    ("Hopf link", lambda: hopf_link_bloch_vector(0.2, k_symbols=(kx, ky, kz)), 48),
    ("Trefoil", lambda: trefoil_bloch_vector(0.25, k_symbols=(kx, ky, kz)), 64),
    ("Solomon", lambda: solomon_bloch_vector(1.0, k_symbols=(kx, ky, kz)), 64),
    ("Three-link", lambda: threelink_bloch_vector(0.41, k_symbols=(kx, ky, kz)), 80),
    ("Torus (1,2)", lambda: pq_torus_knot_bloch_vector(1, 2, 0.5, k_symbols=(kx, ky, kz)), 48),
    ("Torus (3,7)", lambda: pq_torus_knot_bloch_vector(3, 7, 0.2, k_symbols=(kx, ky, kz), c=0.7, m=2.0), 96),
    ("Awesome", lambda: awesome_bloch_vector(0.16, k_symbols=(kx, ky, kz)), 64),
]

energy_gallery_records = []
for name, builder, dimension in energy_gallery_specs:
    ske_energy = NodalSkeleton(
        builder(),
        k_symbols=(kx, ky, kz),
        dimension=dimension,
        axis_scale=(1.0, 1.0, 1.5),
    )
    surface_energy = ske_energy.exceptional_surface_pv.connectivity("largest")
    graph_energy = None
    graph_error = None
    is_planar = None
    try:
        graph_energy = ske_energy.skeleton_graph(simplify=True, smooth_epsilon=2)
        is_planar = nx.check_planarity(nx.Graph(graph_energy))[0]
    except Exception as exc:
        graph_error = f"{type(exc).__name__}: {exc}"
    energy_gallery_records.append((name, surface_energy, graph_energy, graph_error, is_planar))
    print(name)
    print("  surface_points_cells =", (surface_energy.n_points, surface_energy.n_cells))
    if graph_energy is None:
        print("  graph_stage =", graph_error)
    else:
        print("  graph_nodes_edges =", (graph_energy.number_of_nodes(), graph_energy.number_of_edges()))
        print("  planar =", is_planar)


Unknot
  surface_points_cells = (998, 1996)
  graph_nodes_edges = (1, 1)
  planar = True


Hopf link
  surface_points_cells = (1456, 2912)
  graph_nodes_edges = (1, 1)
  planar = True


Trefoil
  surface_points_cells = (4159, 8308)
  graph_nodes_edges = (3, 5)
  planar = True


Solomon
  surface_points_cells = (13244, 26428)
  graph_nodes_edges = (2, 5)
  planar = True


Three-link
  surface_points_cells = (9856, 19724)
  graph_nodes_edges = (6, 9)
  planar = False
Torus (1,2)
  surface_points_cells = (4812, 9636)
  graph_nodes_edges = (6, 9)
  planar = True


Torus (3,7)
  surface_points_cells = (7214, 14424)
  graph_nodes_edges = (1, 1)
  planar = True


Awesome
  surface_points_cells = (3522, 7056)
  graph_nodes_edges = (5, 8)
  planar = True


In [10]:
fig = make_subplots(
    rows=2,
    cols=4,
    specs=[[{"type": "scene"} for _ in range(4)], [{"type": "scene"} for _ in range(4)]],
    horizontal_spacing=0.01,
    vertical_spacing=0.02,
)
for index, (name, surface_energy, graph_energy, graph_error, is_planar) in enumerate(energy_gallery_records):
    row = 1 if index < 4 else 2
    col = index % 4 + 1
    add_surface_trace(fig, surface_energy, row=row, col=col, opacity=0.46)
    if graph_energy is not None:
        add_graph_traces(fig, graph_energy, row=row, col=col)
style_plotly_scenes(fig, 8, width=1120, height=760).show()


### 10.4 Berry-Curvature Slices And Surface-Plane Intersections

The appendix Berry figure used three slicing planes.  The code below exposes
the actual geometric operation: slice the exceptional surface by
`kx=0`, `ky=0`, and `kz=pi/2`, then plot the intersection curves.


In [11]:
berry_appendix_ske = NodalSkeleton(
    hopf_link_bloch_vector(0.8, k_symbols=(kx, ky, kz)),
    k_symbols=(kx, ky, kz),
    dimension=64,
    axis_scale=(1.0, 1.0, 1.5),
)
berry_appendix_surface = berry_appendix_ske.exceptional_surface_pv.connectivity("largest")
berry_slice_specs = [
    ("kx=0", (1, 0, 0), (0, 0, 0), RED),
    ("ky=0", (0, 1, 0), (0, 0, 0), "#2ca02c"),
    ("kz=pi/2", (0, 0, 1), (0, 0, np.pi / 2), "#ff7f0e"),
]
berry_slices = []
for label, normal, origin, color in berry_slice_specs:
    sliced = berry_appendix_surface.slice(normal=normal, origin=origin)
    berry_slices.append((label, sliced, color))
    print(label, "points_cells =", (sliced.n_points, sliced.n_cells))


kx=0 points_cells = (422, 422)
ky=0 points_cells = (376, 376)
kz=pi/2 points_cells = (290, 290)


In [12]:
fig = go.Figure()
mesh = berry_appendix_surface.triangulate()
faces = mesh.faces.reshape(-1, 4)[:, 1:]
pts = mesh.points
fig.add_trace(
    go.Mesh3d(
        x=pts[:, 0],
        y=pts[:, 1],
        z=pts[:, 2],
        i=faces[:, 0],
        j=faces[:, 1],
        k=faces[:, 2],
        color=BLUE,
        opacity=0.22,
        showscale=False,
    )
)
for label, sliced, color in berry_slices:
    for segment in polyline_traces_from_slice(sliced):
        fig.add_trace(
            go.Scatter3d(
                x=segment[:, 0],
                y=segment[:, 1],
                z=segment[:, 2],
                mode="lines",
                line=dict(color=color, width=7),
                showlegend=False,
            )
        )
apply_kg_layout(fig).show()


### 10.5 Intrinsic-Linkedness Minor Check

The appendix included an awesome-surface panel together with an intrinsic
linkedness/Petersen comparison.  The public API exposes this as a graph-minor
question on the extracted skeleton graph.


In [13]:
awesome_minor_ske = NodalSkeleton(
    awesome_bloch_vector(0.2, k_symbols=(kx, ky, kz), c=0.5),
    k_symbols=(kx, ky, kz),
    dimension=120,
    axis_scale=(1.0, 1.0, 1.5),
)
awesome_minor_surface = awesome_minor_ske.exceptional_surface_pv.connectivity("largest")
awesome_minor_graph = awesome_minor_ske.skeleton_graph(simplify=True, smooth_epsilon=2)
petersen = nx.petersen_graph()
minor_embedding = awesome_minor_ske.check_minor(petersen, awesome_minor_graph)

print("awesome surface =", (awesome_minor_surface.n_points, awesome_minor_surface.n_cells))
print("awesome graph =", (awesome_minor_graph.number_of_nodes(), awesome_minor_graph.number_of_edges()))
print("degree sequence =", sorted(dict(awesome_minor_graph.degree()).values()))
print("petersen minor found =", bool(minor_embedding))
if minor_embedding:
    print("embedding sizes =", {node: len(chain) for node, chain in minor_embedding.items()})


The given graph DOES NOT contain the minor graph.
awesome surface = (16718, 33448)
awesome graph = (7, 11)
degree sequence = [3, 3, 3, 3, 3, 3, 4]
petersen minor found = False


In [14]:
fig = make_subplots(
    rows=1,
    cols=2,
    specs=[[{"type": "scene"}, {"type": "xy"}]],
    horizontal_spacing=0.03,
)
add_graph_traces(fig, awesome_minor_graph, row=1, col=1)
pos = nx.spring_layout(petersen, seed=4)
for u, v in petersen.edges():
    fig.add_trace(
        go.Scatter(
            x=[pos[u][0], pos[v][0]],
            y=[pos[u][1], pos[v][1]],
            mode="lines",
            line=dict(color=BLUE, width=3),
            showlegend=False,
        ),
        row=1,
        col=2,
    )
fig.add_trace(
    go.Scatter(
        x=[pos[n][0] for n in petersen.nodes()],
        y=[pos[n][1] for n in petersen.nodes()],
        mode="markers",
        marker=dict(size=10, color=RED),
        showlegend=False,
    ),
    row=1,
    col=2,
)
style_plotly_scenes(fig, 1, width=980, height=520)
fig.update_xaxes(visible=False, row=1, col=2)
fig.update_yaxes(visible=False, row=1, col=2, scaleanchor="x", scaleratio=1)
fig.show()


### 10.6 Three-Link Planarity Evolution

The planarity appendix compared three `gamma` values.  This cell regenerates
all three from the Hamiltonian model, extracts graphs, and checks planarity.


In [15]:
three_link_planarity_records = []
for gamma in (0.116, 0.41, 0.5):
    ske_three = NodalSkeleton(
        threelink_bloch_vector(gamma, k_symbols=(kx, ky, kz), c=0.5),
        k_symbols=(kx, ky, kz),
        dimension=80,
        axis_scale=(1.0, 1.0, 1.5),
    )
    surface_three = ske_three.exceptional_surface_pv.connectivity("largest")
    graph_three = ske_three.skeleton_graph(simplify=True, smooth_epsilon=2)
    planar_three = nx.check_planarity(nx.Graph(graph_three))[0]
    three_link_planarity_records.append((gamma, surface_three, graph_three, planar_three))
    print(f"gamma = {gamma}")
    print("  surface_points_cells =", (surface_three.n_points, surface_three.n_cells))
    print("  graph_nodes_edges =", (graph_three.number_of_nodes(), graph_three.number_of_edges()))
    print("  planar =", planar_three)


gamma = 0.116
  surface_points_cells = (5060, 10128)
  graph_nodes_edges = (2, 4)
  planar = True


gamma = 0.41
  surface_points_cells = (9856, 19724)
  graph_nodes_edges = (6, 9)
  planar = False


gamma = 0.5
  surface_points_cells = (10464, 20932)
  graph_nodes_edges = (2, 3)
  planar = True


In [16]:
fig = make_subplots(
    rows=2,
    cols=3,
    specs=[[{"type": "scene"} for _ in range(3)], [{"type": "scene"} for _ in range(3)]],
    horizontal_spacing=0.01,
    vertical_spacing=0.02,
)
for col, (gamma, surface_three, graph_three, planar_three) in enumerate(three_link_planarity_records, start=1):
    add_surface_trace(fig, surface_three, row=1, col=col, opacity=0.46)
    add_graph_traces(fig, graph_three, row=2, col=col)
style_plotly_scenes(fig, 6, width=1080, height=720).show()
